# 📚 Конспект: Функції та Функціональна Абстракція

> Урок 8 — повний конспект з теорією, чотирма функціональними патернами,
> прикладами та типовими помилками.

---

## 📋 Зміст

| # | Тема | Ключові концепції |
|---|------|-------------------|
| 1 | Навіщо функції? | DRY, Абстракція, Декомпозиція |
| 2 | Ментальна модель виконання | Stack Frame, локальні змінні |
| 3 | Параметри vs Аргументи | сигнатура, виклик |
| 4 | `return` vs `print` | передача даних у програму |
| 5 | Чисті функції | Pure Functions, Side Effects |
| 6 | `*args` | tuple аргументів |
| 7 | Патерн: Предикат | фільтрація, `filter()` |
| 8 | Патерн: Трансформер | `map()`, comprehension |
| 9 | Патерн: Редьюсер | `sum()`, `max()`, `reduce()` |
| 10 | Функціональна Декомпозиція | divide and conquer |
| 11 | Повний Pipeline | Filter → Map → Reduce |
| 12 | Типові помилки | застереження, виправлення |
| 13 | Шпаргалка | швидкий довідник |
| 14 | Задачі | самостійна робота |

---

## 🧠 Як мислить програміст

Програмісти не пишуть довгі послідовності інструкцій — вони проектують **систему перетворення даних**.
Функції — це не просто «згрупований код», це **інструмент управління складністю**.

Коли програміст бачить нову задачу, він ставить чотири питання:

```
1. PREDICATE   — Що відфільтрувати? (Так / Ні)
2. TRANSFORMER — Як змінити форму даних? (один → інший)
3. REDUCER     — Як зібрати у кінцевий результат? (багато → одне)
4. DECOMPOSE   — Як розбити задачу на менші частини?
```

Увесь цей урок — про ці чотири патерни та інструменти Python для їх реалізації.

---

## 🏗️ Частина 1: Навіщо потрібні функції?

### Дві проблеми, які вирішують функції

**Проблема 1 — Дублювання коду (WET: Write Everything Twice)**

```python
# БЕЗ функцій — та сама логіка скрізь
bill1 = 150.0
tax1  = bill1 * 0.20   # ← якщо ставку змінити на 0.22,
final1 = bill1 + tax1  #   треба виправляти у 10 місцях!

bill2 = 300.0
tax2  = bill2 * 0.20
final2 = bill2 + tax2
```

**Проблема 2 — Когнітивне навантаження**

Читаючи 300 рядків коду, мозок не може утримувати всі деталі одночасно.
Функції дозволяють **думати на рівні понять**, а не деталей синтаксису.

### Три рішення, що дають функції

| Проблема | Рішення | Принцип |
|---|---|---|
| Дублювання | Одна функція → багато викликів | **DRY** (Don't Repeat Yourself) |
| Складність | Функція = «чорний ящик» | **Абстракція** |
| Великий розмір | Велика задача → маленькі функції | **Декомпозиція** |

> **Абстракція** — усунення неважливого і посилення суттєвого.
> Знаючи `apply_tax(bill)`, вам не потрібно думати про множення.
> Ви думаєте про **що** відбувається, а не **як**.

In [ ]:
# Без функцій — WET (Write Everything Twice)
bill1 = 150.0
tax1  = bill1 * 0.20
final1 = bill1 + tax1

bill2 = 300.0
tax2  = bill2 * 0.20   # та сама логіка — дублювання!
final2 = bill2 + tax2

print(f"WET: final1={final1}, final2={final2}")
print()

# З функцією — DRY (Don't Repeat Yourself)
def apply_tax(bill, rate=0.20):
    """Повертає суму з податком. Логіка в одному місці."""
    return bill * (1 + rate)

final1 = apply_tax(150.0)
final2 = apply_tax(300.0)
final3 = apply_tax(500.0, rate=0.15)  # іншу ставку — просто змінюємо аргумент

print(f"DRY: final1={final1}, final2={final2}, final3={final3}")
print()

# Тепер хочемо змінити ставку — тільки одне місце!
print("Якщо треба змінити ставку → змінюємо тільки в одному місці: def apply_tax(bill, rate=0.22)")

---

## ⚙️ Частина 2: Ментальна модель виконання — Stack Frame

### Синтаксис функції

```
def  назва_функції  ( параметр1, параметр2 ) :  ← сигнатура
     │                │
  ключове слово    параметри (placeholder-и, НЕ реальні дані)

    # тіло — код, що виконується
    result = параметр1 + параметр2
    return result   ← повертаємо значення
```

### Що відбувається при виклику (Stack Frame)

```
1. ВИКЛИК:   add(5, 10)         ← аргументи передаються в функцію
                ↓
2. СТЕК:     Python створює тимчасовий «ящик» у пам'яті
             ┌─────────────────────┐
             │  a = 5              │ ← локальні змінні живуть тут
             │  b = 10             │
             │  result = 15        │
             └─────────────────────┘
                ↓
3. RETURN:   повертає 15 у головну програму
             «ящик» знищується → a, b, result зникають назавжди
```

> **Важливо:** локальні змінні існують **тільки під час** виконання функції.
> Ззовні вони невидимі — це й називається **ізоляція (scope)**.

In [ ]:
# Демонстрація Stack Frame та ізоляції
def add(a, b):
    result = a + b          # result — локальна змінна
    print(f"  [всередині] a={a}, b={b}, result={result}")
    return result
    # після return: а, b, result зникають


print("Виклик add(5, 10):")
answer = add(5, 10)
print(f"  [ззовні] answer={answer}")
print()

# Локальні змінні ззовні не видно
try:
    print(result)   # ← result не існує ззовні!
except NameError as e:
    print(f"NameError: {e}")
    print("  Локальні змінні ізольовані — ззовні невидимі!")
print()

# Кожен виклик — свій незалежний Stack Frame
print("Два незалежних виклики:")
r1 = add(2, 3)    # свій frame: a=2, b=3
r2 = add(10, 20)  # свій frame: a=10, b=20
print(f"  r1={r1}, r2={r2}")

---

## 🔤 Частина 3: Параметри vs Аргументи

Ці два терміни часто плутають. Вони описують **різні моменти** в житті функції:

```
ПАРАМЕТРИ — імена у визначенні функції (при написанні коду)
              ↓           ↓
def describe(name,      age):
    ...                          ← placeholder-и, «конверти без листів»

АРГУМЕНТИ — реальні дані при виклику (при запуску)
              ↓          ↓
describe("Alice",      25)       ← реальні значення, «листи в конверти»
```

### Типи параметрів

| Тип | Синтаксис | Особливість |
|-----|-----------|-------------|
| Позиційний | `def f(a, b)` | Порядок важливий |
| Зі значенням | `def f(a, b=10)` | Необов'язковий |
| Keyword-only | `def f(a, *, kw)` | Тільки через `kw=значення` |

In [ ]:
# Позиційні параметри — порядок важливий!
def describe(name, age):
    print(f"  {name} має {age} років")

describe("Alice", 25)   # a="Alice", b=25 — правильно
describe(25, "Alice")   # a=25, b="Alice" — семантично неправильно!
print()

# Параметри зі значенням за замовчуванням
def greet(name, greeting="Привіт"):
    print(f"  {greeting}, {name}!")

greet("Олена")              # greeting = "Привіт" за замовчуванням
greet("Іван", "Доброго дня")  # переопределяємо
print()

# Keyword-аргументи — ім'я явно
def power(base, exponent=2):
    return base ** exponent

print(f"  power(3)          = {power(3)}")
print(f"  power(3, 3)       = {power(3, 3)}")
print(f"  power(exponent=3, base=2) = {power(exponent=3, base=2)}")  # keyword — порядок не важливий!

---

## 🖨️ Частина 4: `return` vs `print` — критична різниця

Це одна з найпоширеніших помилок початківців.

```
print()  →  для ЛЮДИНИ   (показує на екрані, програма нічого не отримує)
return   →  для ПРОГРАМИ  (передає значення назад у пайплайн)
```

```python
# ПОМИЛКА: print замість return
def bad_add(a, b):
    print(a + b)     ← показує 5, але програма отримує None

result = bad_add(2, 3)
print(result)        ← виведе: None
result * 10          ← TypeError: unsupported operand type(s) for *: 'NoneType' and 'int'
```

> Якщо функція не має `return` — Python **автоматично** повертає `None`.
> `None` — це «порожнє значення» (не `0`, не `""`, а **відсутність** значення).

In [ ]:
# ПОМИЛКА: print замість return
def bad_add(a, b):
    print(a + b)   # показує 5 на екрані...


# ПРАВИЛЬНО: return
def good_add(a, b):
    return a + b   # передає значення в програму


print("--- bad_add(2, 3) ---")
bad_result = bad_add(2, 3)       # виводить 5 під час виконання
print(f"bad_result  = {bad_result!r}")  # None!

print()
print("--- good_add(2, 3) ---")
good_result = good_add(2, 3)     # нічого не виводить, але зберігає 5
print(f"good_result = {good_result!r}")  # 5
print(f"good_result * 10 = {good_result * 10}")  # 50 — можна використати!

print()
# bad_result * 10 → TypeError
try:
    _ = bad_result * 10
except TypeError as e:
    print(f"bad_result * 10 → TypeError: {e}")
    print("  None не можна множити — функція не повернула значення!")

---

## 🏭 Частина 5: Чисті функції (Pure Functions)

### Два правила чистої функції

```
1. ДЕТЕРМІНОВАНІСТЬ:  однакові аргументи → ЗАВЖДИ однаковий результат
2. БЕЗ ПОБІЧНИХ ЕФЕКТІВ:  не змінює нічого ззовні (no side effects)
```

**Ментальна модель:** ідеальна машина на заводі.
Закладаєте однакову сировину → отримуєте однаковий продукт.
Машина не «забруднює цех» — не змінює глобальні змінні.

### Нечиста vs Чиста — порівняння

```python
# НЕЧИСТА — мутує вхідний список!
def add_value_impure(items, value):
    items.append(value)   ← змінює ОРИГІНАЛ
    return items

# ЧИСТА — створює НОВИЙ список
def add_value_pure(items, value):
    return items + [value]  ← оригінал залишається
```

### Аналогія з вбудованими функціями

| Функція | Тип | Що робить |
|---------|-----|-----------|
| `list.sort()` | Нечиста | Сортує список **на місці** (in-place) |
| `sorted()` | Чиста | Повертає **новий** відсортований список |
| `list.append()` | Нечиста | Додає елемент **в оригінал** |
| `list + [item]` | Чиста | Повертає **новий** список |

In [ ]:
# === Демонстрація: нечиста vs чиста функція ===

# НЕЧИСТА: мутує вхідний список (side effect!)
def apply_discount_impure(prices, discount):
    for i in range(len(prices)):       # змінюємо ОРИГІНАЛЬНИЙ список
        prices[i] = prices[i] * (1 - discount)
    return prices


# ЧИСТА: повертає НОВИЙ список, оригінал не торкається
def apply_discount_pure(prices, discount):
    return [p * (1 - discount) for p in prices]   # новий список!


my_prices = [100.0, 200.0, 300.0]
print(f"До виклику:            {my_prices}")

result = apply_discount_impure(my_prices, 0.10)
print(f"Результат (нечиста):   {result}")
print(f"ОРИГІНАЛ ЗНИЩЕНИЙ:     {my_prices}")    # ← my_prices змінений!
print()

# Чиста функція — безпечна
my_prices2 = [100.0, 200.0, 300.0]
result2 = apply_discount_pure(my_prices2, 0.10)
print(f"Результат (чиста):     {result2}")
print(f"Оригінал збережений:   {my_prices2}")   # ← незмінений!

In [ ]:
# Аналогія: sort() vs sorted()
data = [3, 1, 4, 1, 5, 9, 2, 6]

# ЧИСТА — оригінал збережений
sorted_data = sorted(data)
print(f"sorted() результат: {sorted_data}")
print(f"Оригінал після sorted(): {data}")   # незмінений!
print()

# НЕЧИСТА — оригінал змінений
data.sort()
print(f"Після .sort(): {data}")              # ← оригінал знищено

---

## 📦 Частина 6: `*args` — змінна кількість аргументів

```
def func(*args):
             ↑
   зірочка «пакує» всі позиційні аргументи у TUPLE
```

**Коли використовувати:**
- Кількість аргументів **невідома заздалегідь**
- Функція повинна прийняти **будь-яку кількість** значень

```python
process_sales(150)             → args = (150,)
process_sales(150, 300, 75)    → args = (150, 300, 75)
process_sales(1, 2, 3, 4, 5)  → args = (1, 2, 3, 4, 5)
```

In [ ]:
# *args пакує всі аргументи у tuple
def process_sales(*args):
    print(f"  тип args: {type(args)}")
    print(f"  вміст:    {args}")
    print(f"  кількість: {len(args)}")
    return sum(args)

print("--- Один аргумент ---")
process_sales(150.0)

print("\n--- Три аргументи ---")
total = process_sales(150.0, 300.0, 75.5)
print(f"  Разом: {total}")

print("\n--- П'ять аргументів ---")
total = process_sales(150, -20, 300, 50, -10)
print(f"  Разом: {total}")

In [ ]:
# Практичний приклад: добуток довільної кількості чисел
def calculate_product(*args):
    """Множить усі передані числа між собою."""
    total = 1
    for number in args:      # args — це tuple, ітеруємо по ньому
        total *= number
    return total


print(calculate_product(2, 5))           # 10
print(calculate_product(2, 5, 3))        # 30
print(calculate_product(2, 5, 3, 2))     # 60
print(calculate_product(1, 2, 3, 4, 5))  # 120  ← 5!
print()

# Можна комбінувати *args з іншими параметрами
def weighted_sum(weight, *values):
    """Сума всіх values, помножена на weight."""
    return weight * sum(values)

print(f"weighted_sum(2, 10, 20, 30) = {weighted_sum(2, 10, 20, 30)}")  # 2*(10+20+30) = 120

---

## 🚪 Частина 7: Патерн — Предикат (Predicate / Filter)

**Ментальна модель: «Фейсконтроль»**

```
Охоронець (предикат) дивиться на кожен об'єкт і каже лише:
   ✓ TRUE  — пускає
   ✗ FALSE — не пускає

Предикат НІКОЛИ не змінює дані — тільки оцінює!
```

```
Вхід:  [1, 2, 3, 4, 5, 6, 7, 8]
Тест:  [✗, ✓, ✗, ✓, ✗, ✓, ✗, ✓]
Вихід: [2, 4, 6, 8]
```

### Три способи фільтрації в Python

```python
# 1. filter() — класичний
evens = list(filter(is_even, data))

# 2. list comprehension — пітонічний (найчастіше)
evens = [x for x in data if is_even(x)]

# 3. inline — без окремої функції
evens = [x for x in data if x % 2 == 0]
```

In [ ]:
# Предикати — чисті функції що повертають True/False
def is_even(x):
    """Чи є число парним?"""
    return x % 2 == 0

def is_positive(x):
    """Чи є число додатним?"""
    return x > 0

def is_long_word(word):
    """Чи довше слово за 5 символів?"""
    return len(word) > 5


data = [1, -3, 2, -5, 4, 6, -1, 8]
words = ['cat', 'elephant', 'dog', 'python', 'ant', 'banana']

# Тест предикатів
print(f"is_even(4) = {is_even(4)},  is_even(3) = {is_even(3)}")
print(f"is_positive(-5) = {is_positive(-5)},  is_positive(3) = {is_positive(3)}")
print()

# Три способи фільтрації
evens_v1 = list(filter(is_even, data))               # filter()
evens_v2 = [x for x in data if is_even(x)]          # comprehension + предикат
evens_v3 = [x for x in data if x % 2 == 0]          # inline
print(f"data   = {data}")
print(f"filter()      → {evens_v1}")
print(f"comprehension → {evens_v2}")
print(f"inline        → {evens_v3}")
print()

long_words = [w for w in words if is_long_word(w)]
print(f"words      = {words}")
print(f"long words = {long_words}")

In [ ]:
# Комбінування предикатів — логічні оператори
data = [1, -3, 2, -5, 4, 6, -1, 8]

# AND: парні І додатні
positive_evens = [x for x in data if is_even(x) and is_positive(x)]
print(f"Парні і позитивні: {positive_evens}")

# OR: парні АБО додатні
even_or_positive = [x for x in data if is_even(x) or is_positive(x)]
print(f"Парні або позитивні: {even_or_positive}")

# NOT: непарні
odds = [x for x in data if not is_even(x)]
print(f"Непарні: {odds}")
print()

# all() та any() — редьюсери-предикати
scores = [85, 92, 78, 95, 88]
print(f"scores = {scores}")
print(f"all(s >= 70): {all(s >= 70 for s in scores)}  ← всі >= 70?")
print(f"any(s >= 90): {any(s >= 90 for s in scores)}  ← хоч один >= 90?")
print(f"all(s >= 90): {all(s >= 90 for s in scores)}  ← всі >= 90?")

---

## 🎨 Частина 8: Патерн — Трансформер (Transformer / Map)

**Ментальна модель: «Конвеєр фарбування»**

```
10 білих машин заїжджають → фарбувальна камера → 10 червоних виїжджають

Кількість НЕ змінюється: len(вхід) == len(вихід)
Змінюється тільки ФОРМА/СТАН кожного елемента
```

```
Вхід:  [1, 2, 3, 4, 5]           (5 елементів)
func:   x → x * x
Вихід: [1, 4, 9, 16, 25]         (5 елементів — завжди!)
```

### Generator Expression — лінива трансформація

```python
# List comprehension — всі дані одразу в пам'яті
squares = [x**2 for x in data]      # [1, 4, 9, 16, 25]

# Generator expression — обчислює по одному «на вимогу»
squares = (x**2 for x in data)      # <generator object>
# Ідеально для sum(), max(), min() — не витрачає пам'ять!
```

In [ ]:
# Трансформери — функції що змінюють форму, не видаляють елементи
data = [1, 2, 3, 4, 5]

# 1. map() з lambda
squared_map = list(map(lambda x: x * x, data))

# 2. list comprehension (пітонічний стиль)
squared_comp = [x * x for x in data]

# 3. map() з іменованою функцією
def square(x): return x * x
squared_named = list(map(square, data))

print(f"data            = {data}")
print(f"map(lambda)     = {squared_map}")
print(f"comprehension   = {squared_comp}")
print(f"map(square)     = {squared_named}")
print(f"len(вхід) == len(вихід): {len(data) == len(squared_comp)}  ← завжди!")
print()

# Практичні приклади трансформацій
raw_names = ['  alice ', 'BOB', '  charlie  ']
clean_names = [name.strip().title() for name in raw_names]   # нормалізація
print(f"raw   = {raw_names}")
print(f"clean = {clean_names}")
print()

# Конвертація валюти
prices_usd = [10.5, 25.0, 3.99, 150.0]
rate       = 41.5
prices_uah = [round(p * rate, 2) for p in prices_usd]
print(f"USD: {prices_usd}")
print(f"UAH: {prices_uah}")

In [ ]:
import sys

# List comprehension vs Generator expression — пам'ять
lst_comp = [x**2 for x in range(10_000)]     # будує список в пам'яті
gen_expr = (x**2 for x in range(10_000))     # лінивий генератор

print(f"list comprehension: {sys.getsizeof(lst_comp):>8} байт")
print(f"generator expr    : {sys.getsizeof(gen_expr):>8} байт")
print(f"Різниця: x{sys.getsizeof(lst_comp)//sys.getsizeof(gen_expr):.0f} пам'яті")
print()

# Generator ідеальний для sum() — обчислює по одному
total_squared = sum(x**2 for x in range(10_000))   # gen всередині sum()
print(f"sum(x**2 for x in range(10000)) = {total_squared}")
print("  ↑ Generator не створює проміжний список — економить пам'ять!")

---

## ❄️ Частина 9: Патерн — Редьюсер (Reducer / Aggregation)

**Ментальна модель: «Снігова куля»**

```
Починається маленькою (початкове значення 0)
→ Котиться по масиві даних
→ Наліплює на себе кожен елемент
→ Перехід від БАГАТЬОХ до ОДНОГО

Вхід:  [10, 20, 30, 40]    → 4 елементи
       0+10=10
       10+20=30
       30+30=60
       60+40=100
Вихід: 100                  → одне значення
```

### Вбудовані редьюсери Python

| Функція | Операція | Опис |
|---------|----------|------|
| `sum()` | `0 + x1 + x2 + ...` | Сума |
| `max()` | найбільше | Максимум |
| `min()` | найменше | Мінімум |
| `all()` | AND по всіх | True якщо всі True |
| `any()` | OR по всіх | True якщо хоч один True |
| `functools.reduce()` | будь-яка | Для нестандартних операцій |

In [ ]:
# Ручна реалізація — показуємо механіку
numbers = [10, 20, 30, 40, 50]

print("Крок | Накопичувач | Поточний елемент | Після додавання")
print("-" * 58)

accumulator = 0
for i, n in enumerate(numbers, 1):
    before = accumulator
    accumulator += n
    print(f"  {i}  | {before:11} | {n:16} | {accumulator}")

print(f"\nРезультат: {accumulator}")

In [ ]:
import functools
import operator

data = [10, 20, 30, 40, 50]
print(f"data = {data}")
print()

# Вбудовані редьюсери — переважний стиль
print(f"sum(data)           = {sum(data)}")
print(f"max(data)           = {max(data)}")
print(f"min(data)           = {min(data)}")
print(f"len(data)           = {len(data)}")
print(f"sum/len (середнє)   = {sum(data)/len(data):.1f}")
print()

# Логічні редьюсери
print(f"all(x > 5 for x in data)  = {all(x > 5 for x in data)}   ← всі > 5?")
print(f"any(x > 40 for x in data) = {any(x > 40 for x in data)}  ← хоч один > 40?")
print()

# functools.reduce() — для нестандартних операцій
factorial = functools.reduce(operator.mul, range(1, 6))   # 1*2*3*4*5 = 120
print(f"reduce(mul, 1..5) = {factorial}  ← 5!")
print()

# max/min з ключем сортування
words = ['apple', 'banana', 'fig', 'cherry', 'kiwi']
longest  = max(words, key=len)
shortest = min(words, key=len)
print(f"words   = {words}")
print(f"найдовше: {longest!r}")
print(f"найкоротше: {shortest!r}")

---

## 🔧 Частина 10: Функціональна Декомпозиція

**Стратегія «Розділяй і Володарюй» (Divide and Conquer)**

```
Велика задача
      ↓  декомпозиція
┌─────┴─────┐
│           │
Підзадача1  Підзадача2
│           │
функція1()  функція2()
```

**Принцип єдиної відповідальності:** одна функція = одна задача.
Якщо функцію важко назвати — вона, мабуть, робить забагато.

### Приклад: Знайти середню оцінку тільки тих, хто здав іспит

```
Задача: avg_passed_score(scores)
            ↓
   filter_passed(scores)     ← предикат: score >= 60
   calculate_average(passed) ← редьюсер: sum/len
```

In [ ]:
# === Декомпозиція: маленькі одноцільові функції ===

# Рівень 1: прості предикати
def is_passed(score):
    """Чи здав студент іспит (≥60)?"""
    return score >= 60

def is_excellent(score):
    """Чи відмінна оцінка (≥90)?"""
    return score >= 90


# Рівень 1: прості трансформери
def letter_grade(score):
    """Перетворює числову оцінку на літерну."""
    if score >= 90: return 'A'
    if score >= 80: return 'B'
    if score >= 70: return 'C'
    if score >= 60: return 'D'
    return 'F'


# Рівень 1: прості редьюсери
def safe_average(values):
    """Середнє значення, захищене від порожнього списку."""
    if not values:
        return 0.0
    return sum(values) / len(values)


# Рівень 2: функції що компонують прості
def avg_passed_score(scores):
    """Середня оцінка тільки тих, хто здав."""
    passed = [s for s in scores if is_passed(s)]    # filter
    return safe_average(passed)                      # reduce


def class_report(scores):
    """Повний звіт по класу."""
    passed    = [s for s in scores if is_passed(s)]
    failed    = [s for s in scores if not is_passed(s)]
    grades    = [letter_grade(s) for s in scores]      # map
    return {
        'total':    len(scores),
        'passed':   len(passed),
        'failed':   len(failed),
        'avg_all':  safe_average(scores),
        'avg_pass': safe_average(passed),
        'max':      max(scores) if scores else 0,
        'grades':   grades,
    }


# Використання
scores = [55, 82, 78, 92, 45, 88, 63, 70, 50, 95]
print(f"Оцінки: {scores}")
print(f"Середня серед здали: {avg_passed_score(scores):.1f}")
print()

report = class_report(scores)
print("Звіт по класу:")
for key, val in report.items():
    print(f"  {key:12}: {val}")

---

## 🔄 Частина 11: Повний Pipeline — Filter → Map → Reduce

**Ідея пайплайну:** дані «протікають» через ланцюжок чистих функцій.

```
Сирі дані
    │
    ├─ Step 1: Predicate/Filter — відкинути зайве
    │
    ├─ Step 2: Transformer/Map  — змінити форму
    │
    └─ Step 3: Reducer/Reduce   — зібрати відповідь
```

**`*args` + generator = ефективний пайплайн:**
```python
def pipeline(*args):
    taxed = (sale * 1.10 for sale in args if sale > 0)  # filter + map у генераторі
    return sum(taxed)   # reduce — «тягне» дані з генератора по одному
```
- `sum()` — **консьюмер** (споживач), тягне дані
- Генератор — **продьюсер** (виробник), дає по одному

In [ ]:
# === Пайплайн з *args + generator ===

def process_sales_pipeline(*args):
    """
    Обчислює виручку з ПДВ тільки від позитивних продажів.

    *args = (150, -20, 300, 50, -10)
    Step 1 (Filter):  відкидаємо від'ємні
    Step 2 (Map):     застосовуємо ПДВ 10%
    Step 3 (Reduce):  сумуємо
    """
    taxed_gen = (sale * 1.10 for sale in args if sale > 0)
    #             ↑ MAP                ↑ FILTER
    return sum(taxed_gen)   # ← REDUCE


result = process_sales_pipeline(150, -20, 300, 50, -10)
print(f"Виручка з ПДВ (тільки позитивні): ${result:.2f}")
# Розрахунок: (150 + 300 + 50) * 1.10 = 550.0
print()

# Покроково — для розуміння
raw = (150, -20, 300, 50, -10)
step1 = [s for s in raw if s > 0]          # Filter
step2 = [s * 1.10 for s in step1]          # Map
step3 = sum(step2)                          # Reduce

print(f"Вхід:           {raw}")
print(f"Крок 1 (Filter): {step1}")
print(f"Крок 2 (Map):    {[round(x,2) for x in step2]}")
print(f"Крок 3 (Reduce): {step3:.2f}")

In [ ]:
# === Повний реальний пайплайн: аналіз оцінок класу ===

raw_data = [
    "Alice,92", "Bob,ERROR", "Charlie,85",
    "Diana,",   "Eve,78",    "Frank,65",
    "Grace,95", "Henry,71",  "Ivan,45",
]

print(f"Сирі дані: {len(raw_data)} рядків\n")

# Маленькі чисті функції
def is_valid_row(row):
    """Предикат: рядок має дві частини та числову оцінку."""
    parts = row.split(',')
    return len(parts) == 2 and parts[1].strip().isdigit()

def parse_row(row):
    """Трансформер: рядок → словник."""
    name, score = row.split(',')
    return {'name': name.strip(), 'score': int(score.strip())}

def add_grade(record):
    """Трансформер: додає літерну оцінку."""
    s = record['score']
    grade = 'A' if s>=90 else 'B' if s>=80 else 'C' if s>=70 else 'D' if s>=60 else 'F'
    return {**record, 'grade': grade}   # {**d, key: val} — копія з додатковим полем


# PIPELINE
# Step 1: Filter — тільки валідні рядки
valid_rows = [r for r in raw_data if is_valid_row(r)]

# Step 2: Map — парсимо у словники
records = [parse_row(r) for r in valid_rows]

# Step 3: Map — додаємо літерні оцінки
enriched = [add_grade(r) for r in records]

# Step 4: Filter — тільки ті хто здав (не F)
passed = [r for r in enriched if r['grade'] != 'F']

# Step 5: Reduce — агрегуємо
avg_all  = sum(r['score'] for r in enriched) / len(enriched)
avg_pass = sum(r['score'] for r in passed) / len(passed)

# Результат
print(f"Step 1 (Filter valid): {len(valid_rows)}/{len(raw_data)} рядків")
print(f"Step 5 (Passed):       {len(passed)}/{len(enriched)} студентів")
print(f"Середня по всіх:       {avg_all:.1f}")
print(f"Середня по здавших:    {avg_pass:.1f}")
print()
print("Таблиця студентів:")
print("-" * 32)
for r in enriched:
    marker = "✓" if r['grade'] != 'F' else "✗"
    print(f"  {marker} {r['name']:10}: {r['score']:3}  ({r['grade']})")

---

## ⚠️ Частина 12: Типові помилки — Застереження

### Застереження 1: `print` замість `return`
```python
# ❌ Функція нічого не повертає!
def add(a, b):
    print(a + b)      ← тільки показує на екрані

result = add(2, 3)   # result = None
result * 10          # TypeError!

# ✅ Повертаємо значення
def add(a, b):
    return a + b
```

### Застереження 2: Мутація вхідних даних (нечиста функція)
```python
# ❌ Знищує оригінал!
def add_item(lst, item):
    lst.append(item)   ← мутує оригінал
    return lst

# ✅ Чиста — зберігає оригінал
def add_item(lst, item):
    return lst + [item]  ← новий список
```

### Застереження 3: Глобальні змінні всередині функції
```python
# ❌ Прихована залежність — важко відлагодити
total = 0
def add_to_total(x):
    global total
    total += x   ← side effect!

# ✅ Параметри та return — явно
def add_to_total(current_total, x):
    return current_total + x
```

### Застереження 4: Надто велика функція (порушення декомпозиції)
```
❌ Якщо функція робить більше однієї речі — розбийте її!
   def process_and_validate_and_save():   ← три речі!

✅ Три окремі функції:
   def validate(data): ...
   def process(data): ...
   def save(data): ...
```

In [ ]:
# === Демонстрація застереження 2: мутація ===

# НЕЧИСТА — небезпечна!
def add_item_bad(lst, item):
    lst.append(item)   # мутує оригінал
    return lst

# ЧИСТА — безпечна
def add_item_good(lst, item):
    return lst + [item]   # новий список


original = [1, 2, 3]

result_bad = add_item_bad(original, 99)
print(f"нечиста: результат = {result_bad}")
print(f"нечиста: ОРИГІНАЛ ЗНИЩЕНИЙ = {original}")

print()

original2 = [1, 2, 3]
result_good = add_item_good(original2, 99)
print(f"чиста: результат = {result_good}")
print(f"чиста: оригінал збережено = {original2}")
print()

# === Застереження 3: global — антипатерн ===
print("=== Глобальні змінні — антипатерн ===")

total = 0   # глобальна — небезпечна

def bad_accumulate(x):
    global total       # прихована залежність!
    total += x

def good_accumulate(current, x):
    return current + x   # явно — без сюрпризів

bad_accumulate(10)
bad_accumulate(20)
print(f"bad_accumulate: total={total}  ← глобальна змінна змінена")

# Чистий варіант
running_total = 0
running_total = good_accumulate(running_total, 10)
running_total = good_accumulate(running_total, 20)
print(f"good_accumulate: total={running_total}  ← явний контроль")

---

## 📋 Частина 13: Шпаргалка (Quick Reference)

### Синтаксис
```python
def func_name(param1, param2=default, *args):   # визначення
    """Docstring — опис функції."""              # документація
    result = param1 + param2
    return result                                # передаємо в програму

answer = func_name(10, 20)                       # виклик
```

### Чотири патерни
```python
# PREDICATE (True/False)
def is_valid(x): return x > 0
valid = [x for x in data if is_valid(x)]   # filter
valid = list(filter(is_valid, data))        # альтернатива

# TRANSFORMER (один → інший)
def transform(x): return x * 1.10
mapped = [transform(x) for x in data]     # map
mapped = list(map(transform, data))        # альтернатива

# REDUCER (багато → одне)
total = sum(data)                          # сума
top   = max(data, key=func)               # максимум за ключем
check = all(x > 0 for x in data)          # всі?
found = any(x > 100 for x in data)        # хоч один?

# DECOMPOSE (велике → маленьке)
def step1_filter(data): ...
def step2_map(data): ...
def step3_reduce(data): ...
def full_pipeline(data):
    return step3_reduce(step2_map(step1_filter(data)))
```

### *args
```python
def func(*args):          # args — це tuple
    total = sum(args)     # ітеруємо по tuple
    return total

func(1, 2, 3)             → args = (1, 2, 3)
func(*[1, 2, 3])          → те саме, розпакування списку
```

### Чисті функції
```python
# ✅ ЧИСТА: детерміновна + без side effects
def pure(lst): return sorted(lst)         # новий список

# ❌ НЕЧИСТА: мутує вхід або залежить від зовнішнього стану
def impure(lst): lst.sort(); return lst   # in-place!
```

---

# 📝 Задачі — Самостійна робота

### TODO 1: Чиста функція vs Нечиста

Дано список цін. Напишіть **чисту функцію** `apply_vat(prices, rate=0.20)`,
яка повертає **новий** список цін з ПДВ.
Перевірте що оригінал залишається незмінним.

In [ ]:
prices = [100.0, 250.0, 75.0, 400.0]

# YOUR CODE HERE
def apply_vat(prices, rate=0.20):
    pass   # замініть на реалізацію

result = apply_vat(prices)
print(f"Результат: {result}")
print(f"Оригінал:  {prices}")   # має бути незмінним!

# Очікуваний вивід:
# Результат: [120.0, 300.0, 90.0, 480.0]
# Оригінал:  [100.0, 250.0, 75.0, 400.0]

### TODO 2: Предикат + Фільтрація

Дано список слів. Напишіть предикат `is_palindrome(word)` (True якщо слово-паліндром).
Знайдіть усі палідроми у списку.

In [ ]:
words = ['level', 'python', 'radar', 'hello', 'civic', 'world', 'madam', 'noon']

# YOUR CODE HERE
def is_palindrome(word):
    pass   # Підказка: word == word[::-1]

# Знайдіть усі палідроми
# palindromes = ...

# Очікуваний вивід:
# ['level', 'radar', 'civic', 'madam', 'noon']

### TODO 3: *args Редьюсер

Напишіть функцію `sales_stats(*amounts)`, яка приймає
довільну кількість сум і повертає словник:
`{'count': ..., 'total': ..., 'average': ..., 'max': ...}`

In [ ]:
# YOUR CODE HERE
def sales_stats(*amounts):
    pass   # amounts — це tuple всередині функції

stats = sales_stats(150, 300, 75, 200, 450)
print(stats)

# Очікуваний вивід:
# {'count': 5, 'total': 1175, 'average': 235.0, 'max': 450}

### TODO 4: Трансформер Pipeline

Дано список рядків з числами, деякі — з помилками.
Побудуйте пайплайн:
1. Відфільтруйте валідні (`.isdigit()`)
2. Перетворіть у числа (`int`)
3. Залишіть тільки чотиризначні (1000–9999)
4. Відсортуйте за спаданням
5. Виведіть середнє

In [ ]:
raw = ['1234', 'abc', '5678', '99', 'ERROR', '4321', '10000', '8888', '', '7654']

# YOUR CODE HERE
# Step 1: filter valid strings
# Step 2: map to int
# Step 3: filter 4-digit
# Step 4: sort descending
# Step 5: calculate average

# Очікуваний вивід:
# Відфільтровано: [8888, 7654, 5678, 4321, 1234]
# Середнє: 5515.0

### TODO 5 ⭐: Повна декомпозиція

Дано список транзакцій: `(категорія, сума)`.
Використовуючи декомпозицію (маленькі чисті функції), напишіть:
1. `filter_positive(txs)` — тільки позитивні суми
2. `group_by_category(txs)` — `{категорія: [суми]}`
3. `category_totals(groups)` — `{категорія: сума}`
4. `top_category(totals)` — категорія з найбільшою сумою
5. Зберіть у `full_report(txs)` що викликає всі чотири

In [ ]:
from collections import defaultdict

transactions = [
    ('Food', 150), ('Transport', 30), ('Food', -20),
    ('Books', 80), ('Food', 200),     ('Transport', -5),
    ('Books', 120), ('Food', 90),     ('Transport', 45),
]

# YOUR CODE HERE
def filter_positive(txs): pass
def group_by_category(txs): pass
def category_totals(groups): pass
def top_category(totals): pass
def full_report(txs): pass

# Очікуваний вивід:
# Категорії: {'Food': 440, 'Transport': 75, 'Books': 200}
# Топ-категорія: Food